In [ ]:
# ============================================
# PROJECT: Saudi Retail Sales Forecasting
# DATA SOURCE: SAMA (Saudi Central Bank)
# TOOLS: Python + SQL + Power BI
# AUTHOR: Your Name
# DATE: 2026
# ============================================

# --- Install Prophet ---
!pip install prophet --quiet

# --- Import all libraries ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# --- Display settings ---
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print("✅ All libraries imported successfully!")
print("🚀 Project: Saudi Retail Sales Forecasting")
print("📊 Tools: Python | SQL | Power BI | Prophet")

✅ All libraries imported successfully!
🚀 Project: Saudi Retail Sales Forecasting
📊 Tools: Python | SQL | Power BI | Prophet


# STEP 2: Load the SAMA Dataset

In [ ]:
# ============================================
# STEP 2: Load the SAMA Dataset
# ============================================

# Load the file
df = pd.read_parquet("/content/pos-transactions.parquet")

# Basic info
print("✅ Dataset Loaded Successfully!")
print(f"📦 Total Rows: {df.shape[0]}")
print(f"📦 Total Columns: {df.shape[1]}")
print(f"\n📋 Column Names:")
for col in df.columns:
    print(f"   → {col}")
print(f"\n📅 Sample Data:")
df.head()

✅ Dataset Loaded Successfully!
📦 Total Rows: 2333
📦 Total Columns: 8

📋 Column Names:
   → periodicity
   → year
   → quarter
   → month
   → indicator
   → value_in_different_units
   → date
   → date_object

📅 Sample Data:


,periodicity,year,quarter,month,indicator,value_in_different_units,date,date_object
0,Annually,2021-01-01,None,None,Total POS : Sales (In Thousand Riyals),"473,258,165.58",2021,2021-01-01
1,Annually,2024-01-01,None,None,Total POS : Sales (In Thousand Riyals),"668,184,480.83",2024,2024-01-01
2,Annually,1996-01-01,None,None,Total POS : Number of Transactions,"6,834,075.00",1996,1996-01-01
3,Annually,2001-01-01,None,None,Total POS : Number of Transactions,"23,962,839.00",2001,2001-01-01
4,Annually,2002-01-01,None,None,Total POS : Number of Transactions,"33,203,974.00",2002,2002-01-01


#STEP 3: Clean & Filter Data

In [ ]:
# 3a: Keep MONTHLY data only
df_monthly = df[df['periodicity'] == 'Monthly']
print(f"✅ Monthly rows: {df_monthly.shape[0]}")

# 3b: Keep SALES indicator only
df_sales = df_monthly[
    df_monthly['indicator'].str.contains('Total POS :  Sales', na=False)
]
print(f"✅ Sales rows: {df_sales.shape[0]}")

# 3c: Keep only needed columns
df_clean = df_sales[['date', 'value_in_different_units']].copy()

# 3d: Rename columns
df_clean.columns = ['date', 'sales_thousand_sar']

# 3e: Fix date format
df_clean['date'] = pd.to_datetime(df_clean['date'])

# 3f: Filter from 2015 onwards
df_clean = df_clean[df_clean['date'] >= '2015-01-01']

# 3g: Sort by date
df_clean = df_clean.sort_values('date').reset_index(drop=True)

# 3h: Add helper columns
df_clean['year']       = df_clean['date'].dt.year
df_clean['month']      = df_clean['date'].dt.month
df_clean['month_name'] = df_clean['date'].dt.strftime('%B')
df_clean['quarter']    = df_clean['date'].dt.quarter

# 3i: Check missing values
print(f"\n🔍 Missing Values: {df_clean.isnull().sum().sum()}")
print(f"📅 Date Range: {df_clean['date'].min()} → {df_clean['date'].max()}")
print(f"✅ Final Clean Shape: {df_clean.shape}")
print(f"\n📋 Clean Data Preview:")
df_clean.head(10)

✅ Monthly rows: 1647
✅ Sales rows: 375

🔍 Missing Values: 0
📅 Date Range: 2015-01-01 00:00:00 → 2026-03-01 00:00:00
✅ Final Clean Shape: (135, 6)

📋 Clean Data Preview:


,date,sales_thousand_sar,year,month,month_name,quarter
0,2015-01-01,"14,226,961.19",2015,1,January,1
1,2015-02-01,"15,352,750.69",2015,2,February,1
2,2015-03-01,"15,401,083.04",2015,3,March,1
3,2015-04-01,"13,566,169.77",2015,4,April,2
4,2015-05-01,"15,145,620.36",2015,5,May,2
5,2015-06-01,"16,063,976.29",2015,6,June,2
6,2015-07-01,"14,467,041.48",2015,7,July,3
7,2015-08-01,"14,671,878.82",2015,8,August,3
8,2015-09-01,"12,548,198.35",2015,9,September,3
9,2015-10-01,"12,542,228.43",2015,10,October,4


In [ ]:
# ================================================
# SAVE CORRECT CSV — Run this in Notebook 1
# After your cleaning cell (Image 3)
# ================================================
import os

# The clean data is already in df_clean from Image 3
# Just save it with the correct name

df_clean.to_csv("/content/saudi_sales_clean.csv", index=False)

# Verify
import pandas as pd
check = pd.read_csv("/content/saudi_sales_clean.csv")
print(f"✅ Saved correctly!")
print(f"📊 Rows: {check.shape[0]}")
print(f"📋 Columns: {check.columns.tolist()}")
print(f"📅 First date: {check['date'].iloc[0]}")
print(f"📅 Last date:  {check['date'].iloc[-1]}")

✅ Saved correctly!
📊 Rows: 135
📋 Columns: ['date', 'sales_thousand_sar', 'year', 'month', 'month_name', 'quarter']
📅 First date: 2015-01-01
📅 Last date:  2026-03-01
